In [1]:
using Distributed
addprocs(2)

2-element Vector{Int64}:
 2
 3

In [2]:
@everywhere using QuantumClifford
@everywhere using Random
@everywhere using AbstractAlgebra
using NPZ
using Serialization

In [124]:
@everywhere function seqCircuit_monitored!(Na, Nb, T, Us)
    Random.seed!(abs(hash(myid())) + time_ns())

    # maximally entangled state preparation
    state = one(Destabilizer, 2*Na+Nb)
    for i in 1:Na
        apply!(state, sHadamard(i))            # Apply Hadamard on left qubit
        apply!(state, sCNOT(i, i+Na))       # CNOT between left and right qubit
    end
    
    for t in 1:T
        # random clifford unitary
        apply!(state, Us[t], collect(Na+1: 2*Na+Nb))
        # measurement on bath
        for x in 2*Na+1: 2*Na+Nb
            _, bit = projectZrand!(state, x)
            if bit == 2
                apply!(state, sX(x))   # flip if measured 1
            end
        end
    end
    return state
end

In [ ]:
@everywhere function seqCircuit_monitored_lessR!(Nr, Na, Nb, T, Us)
    # assuming Nr < Na
    Random.seed!(abs(hash(myid())) + time_ns())

    # maximally entangled state over 2^Nr basis preparation
    state = one(Destabilizer, Nr+Na+Nb)
    for i in 1:Nr
        apply!(state, sHadamard(i))            # Apply Hadamard on left qubit
        apply!(state, sCNOT(i, i+Nr))       # CNOT between left and right qubit
    end

    for t in 1:T
        # random clifford unitary
        apply!(state, Us[t], collect(Nr+1: Nr+Na+Nb))
        # measurement on bath
        for x in Nr+Na+1: Nr+Na+Nb
            _, bit = projectZrand!(state, x)
            if bit == 2
                apply!(state, sX(x))   # flip if measured 1
            end
        end
    end
    return state
end


(2, 2)

In [168]:
@everywhere function seqCircuit_unmonitored!(Na, Nb, T, Us)
    # maximally entangled state preparation
    state = MixedDestabilizer(one(Destabilizer, 2*Na+Nb))
    for i in 1:Na
        apply!(state, sHadamard(i))            # Apply Hadamard on left qubit
        apply!(state, sCNOT(i, i+Na))       # CNOT between left and right qubit
    end
    
    bath_one = one(Destabilizer, Nb)
    for t in 1:T
        # random clifford unitary
        apply!(state, Us[t], collect(Na+1: 2*Na+Nb))
        # traceout bath
        traceout!(state, collect(2*Na+1: 2*Na+Nb))
        # reset the bath
        if t < T
            reset_qubits!(state, bath_one, collect(2*Na+1: 2*Na+Nb))
        end
    end
    return state
end

In [ ]:
@everywhere function seqCircuit_unmonitored_lessR!(Nr, Na, Nb, T, Us)
    # maximally entangled state over 2^Nr basis preparation
    state = MixedDestabilizer(one(Destabilizer, Nr+Na+Nb))
    for i in 1:Nr
        apply!(state, sHadamard(i))            # Apply Hadamard on left qubit
        apply!(state, sCNOT(i, i+Nr))       # CNOT between left and right qubit
    end
    
    bath_one = one(Destabilizer, Nb)
    for t in 1:T
        # random clifford unitary
        apply!(state, Us[t], collect(Nr+1: Nr+Na+Nb))
        # traceout bath
        traceout!(state, collect(Nr+Na+1: Nr+Na+Nb))
        # reset the bath
        if t < T
            reset_qubits!(state, bath_one, collect(Nr+Na+1: Nr+Na+Nb))
        end
    end
    return state
end

In [ ]:
@everywhere function seqCircuit_monitored_doped!(Na, Nb, Nc, tdope, T, Us)
    Random.seed!(abs(hash(myid())) + time_ns())

    # maximally entangled state preparation
    state = MixedDestabilizer(one(Destabilizer, 2*Na+Nb+Nc))
    for i in 1:Na
        apply!(state, sHadamard(i))            # Apply Hadamard on left qubit
        apply!(state, sCNOT(i, i+Na))       # CNOT between left and right qubit
    end
    
    bath_one = one(Destabilizer, Nb)
    for t in 1:T
        # random clifford unitary
        apply!(state, Us[t], collect(Na+1: 2*Na+Nb+Nc))
        if t % tdope != 0
            # measurement on all bath
            for x in 2*Na+1: 2*Na+Nb+Nc
                _, bit = projectZrand!(state, x)
                if bit == 2
                    apply!(state, sX(x))   # flip if measured 1
                end
            end
        else
            # measurement on bath B
            for x in 2*Na+1: 2*Na+Nb
                _, bit = projectZrand!(state, x)
                if bit == 2
                    apply!(state, sX(x))   # flip if measured 1
                end
            end
            # traceout the bath C
            traceout!(state, collect(2*Na+Nb+1: 2*Na+Nb+Nc))
            # reset the bath C
            if t < T
                reset_qubits!(state, bath_one, collect(2*Na+Nb+1: 2*Na+Nb+Nc))
            end            
        end
    end
    return state
end

In [75]:
@everywhere function bipartite_entanglement!(state, A)
    """
    given a state, return the bipartite bipartite_entanglement
    of a contiguous subregion A
    Args:
        state: quantum state described in tableau
        A: list of qubit index
    """
    # transform in clipped gauge
    state_clip = canonicalize_clip!(stabilizerview(state))
    # obtain the bigram of the stabilizer generator set
    B_G = bigram(state_clip)
    # count the number of generators with l(g), r(g) \in A
    G_A_size = count(i -> (B_G[i, 1] in A) && (B_G[i, 2] in A), 1:size(B_G, 1))

    return length(A) - G_A_size
end

In [ ]:
Na = 32
Nb = 16
Ts = 2 .^ (1:8)

for T in Ts
    t_begin = time()
    EEs = zeros(50, 100)
    for k in 1:50
        Us = [random_clifford(Na+Nb) for _ in 1:T]

        inputs = [(Na, Nb, T, Us) for i in 1:100]
        states = pmap(x -> seqCircuit_monitored!(x...), inputs)

        inputs = [(states[i], 1:Na) for i in 1:100]
        EEs[k, 1:100] = pmap(x -> bipartite_entanglement!(x...), inputs)
    end
    println("Elapsed time for $(Na) qubits $(T) steps is $(time()-t_begin) seconds")
    npzwrite("data/seqModel/clifford/Na$(Na)/monitored/EEhalf_Na$(Na)Nb$(Nb)T$(T).npy", EEs)
end

Elapsed time for 10 qubits 32 steps is 45.085999965667725 seconds


In [ ]:
Na = 16
Nb = Na ÷ 4
Ts = collect(1:12)

MIs = zeros(length(Ts)+1, 50)
MIs[1, 1:50] .= 2*Na
for T in Ts
    t_begin = time()
    Us = [[random_clifford(Na+Nb) for _ in 1:T] for _ in 1:50]
    
    inputs = [(Na, Nb, T, Us[i]) for i in 1:50]
    states = pmap(x -> seqCircuit_unmonitored!(x...), inputs)

    inputs = [(states[i], 1:Na, Na+1:2*Na) for i in 1:50]
    MIs[T+1, 1:50] = pmap(x -> mutual_information!(x...), inputs)

    println("Elapsed time for $(Na) qubits $(T) steps is $(time()-t_begin) seconds")
end
npzwrite("data/seqModel/clifford/Na$(Na)/traceout/MI_Na$(Na)Nb$(Nb).npy", MIs)

In [ ]:
Nr = 118
Na = 128
Nb = 16
Ts = collect(1:20)

MIs = zeros(length(Ts)+1, 50)
MIs[1, 1:50] .= 2*Nr
for k in 1:length(Ts)
    T = Ts[k]
    t_begin = time()
    Us = [[random_clifford(Na+Nb) for _ in 1:T] for _ in 1:50]
    
    inputs = [(Nr, Na, Nb, T, Us[i]) for i in 1:50]
    states = pmap(x -> seqCircuit_unmonitored_lessR!(x...), inputs)

    inputs = [(states[i], 1:Nr, Nr+1:Nr+Na) for i in 1:50]
    MIs[k+1, 1:50] = pmap(x -> mutual_information!(x...), inputs)

    println("Elapsed time for $(Na) qubits $(T) steps is $(time()-t_begin) seconds")
end
npzwrite("data/seqModel/clifford/Na$(Na)/traceout/MI_Nr$(Nr)Na$(Na)Nb$(Nb).npy", MIs)

In [ ]:
Na = 128
Nb = 16
Nc = 16
tdope = 64
Ts = collect(10:10:128)

MIs = zeros(length(Ts), 50, 100)
for j in 1:length(Ts)
    T = Ts[j]
    t_begin = time()
    for k in 1:50
        Us = [random_clifford(Na+Nb+Nc) for _ in 1:T]

        inputs = [(Na, Nb, Nc, tdope, T, Us) for i in 1:100]
        states = pmap(x -> seqCircuit_monitored_doped!(x...), inputs)

        inputs = [(states[i], 1:Na, Na+1:2*Na) for i in 1:100]
        MIs[j, k, 1:100] = pmap(x -> mutual_information!(x...), inputs)
    end
    println("Elapsed time for $(Na) qubits $(T) steps is $(time()-t_begin) seconds")
end
npzwrite("data/seqModel/clifford/Na$(Na)/monitorDoped/MI_Na$(Na)Nb$(Nb)Nc$(Nc)s$(tdope).npy", MIs)